# Products Analysis

#### Products KPIs
- Top & low performers products (Product concentration)
- Category/subcategory performance
- Average order quantity per product
- price vs volume relationship
- 3 months forcast
- crosselling 

In [110]:
import pandas as pd

import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly.subplots import make_subplots

import numpy as np

In [111]:
clean_transactions=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_transactions.csv")
clean_orders=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_orders.csv")
clean_customers=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_customers.csv")
clean_catalogue=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_catalogue.csv")


In [112]:
Valid_sales=clean_orders[clean_orders["order_status"].isin(["Terminée", "Partiellement remboursée"])]

# Top vs Declining products

In [114]:
def Top_products(clean_transactions, top_n=10):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]
    top_products=(
        ok_transactions.groupby("product_name").agg(
                total_revenue=("line_total", "sum"),
                total_orders=("order_id_stage", "count")
            ).reset_index().sort_values(by="total_revenue", ascending=False).head(top_n)
        )
    return top_products

In [115]:
def low_performers_products(clean_transactions, bottom_n=10):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]
    low_performers_products=(
        ok_transactions.groupby("product_name").agg(
                total_revenue=("line_total", "sum"),
                total_orders=("order_id_stage", "count")
            ).reset_index().sort_values(by="total_revenue", ascending=True).head(bottom_n)
    )
    return low_performers_products

In [116]:
Top_products(clean_transactions, top_n=10)

,product_name,total_revenue,total_orders
59,Chaudière SL-DL32,21576893.72,73
62,Chaudière SL-DM24,16800699.03,128
63,Chaudière SL-DM28,12076085.65,85
60,Chaudière SL-DL36,8664460.00,58
61,Chaudière SL-DM18,8315023.75,82
298,Moniteur 2 fils 4.3 pouces VFE11,6117174.03,75
228,Lampe led éclairage public 50W-6.5K-360°-AC LE...,6113941.64,33
427,Régulateur B25-21mbar,3826784.21,16
57,Chaudière SL-DE36,3483398.00,20
131,Disjoncteur différentiel sensible 30mA 1P+N DD...,2753003.87,19


In [ ]:
low_performers_products(clean_transactions, bottom_n=10)

## Category/subcategory performance


In [ ]:
def performance_per_category(clean_transactions):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]

    top_categories = (
        ok_transactions
        .groupby("category")
        .agg(
            total_revenue=("line_total", "sum"),
            total_orders=("order_id_stage", "nunique")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )
    return top_categories
    

In [ ]:
performance_per_category(clean_transactions)

In [ ]:
def performance_per_subcategory(clean_transactions):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]
    ok_transactions = ok_transactions.drop(columns=["category", "subcategory"])

    merged = ok_transactions.merge(
        clean_catalogue[["sku", "category", "subcategory"]],
        on="sku",
        how="left"
    )
    top_subcategory = (
        merged
        .groupby("subcategory")
        .agg(
            total_revenue=("line_total", "sum"),
            total_orders=("order_id_stage", "nunique")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )
    return top_subcategory
    

In [ ]:
performance_per_subcategory(clean_transactions)

## Average order quantity per product


In [ ]:
def avg_order_quant_per_product(clean_transactions):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]

    avg_quantities = (
        ok_transactions
        .groupby("product_name")
        .agg(
            total_units_sold=("quantity", "sum"),
            avg_quantity_per_order=("quantity", "mean"),
            number_of_orders=("order_id_stage", "nunique")
        )
        .reset_index()
    )
    return avg_quantities

In [ ]:
avg_order_quant_per_product(clean_transactions)

## price vs volume relationship


In [ ]:
def price_vs_volume(clean_transactions):
    ok_transactions = clean_transactions[clean_transactions["sku_quality"] == "ok"]
    price_volume = (
        ok_transactions
        .groupby("product_name")
        .agg(
            total_units_sold=("quantity", "sum"),
            avg_price=("unit_price", "mean"),
            total_revenue=("line_total", "sum")
        )
        .reset_index()
    )
    return price_volume

In [ ]:
price_vs_volume(clean_transactions)

## Graphs

In [119]:
topproducts= Top_products(clean_transactions, top_n=10)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("total_revenue", "total_orders")
)

fig.add_trace(
    go.Bar(x=topproducts["product_name"], y=topproducts["total_revenue"], name="total_revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=topproducts["product_name"], y=topproducts["total_orders"], name="total_orders"),
    row=1, col=2
)
fig.update_layout(title_text="Top Products", showlegend=False)
fig.show()